In [10]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor

# Example dataset
data = pd.DataFrame({
    'feature1': np.random.rand(100),
    'feature2': np.random.rand(100),
    'target': np.random.rand(100)
})

# Split data into training and testing sets
X = data[['feature1', 'feature2']]
y = data['target']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [11]:
# Define models in a tidy-like format
models = pd.DataFrame({
    'model_name': ['Linear Regression', 'Random Forest', 'XGBoost'],
    'model': [LinearRegression(), RandomForestRegressor(random_state=42), XGBRegressor(random_state=42)]
})

# Display the models DataFrame
print(models)

          model_name                                              model
0  Linear Regression                                 LinearRegression()
1      Random Forest             RandomForestRegressor(random_state=42)
2            XGBoost  XGBRegressor(base_score=None, booster=None, ca...


In [12]:
def evaluate_model(model, X_train, X_test, y_train, y_test):
    # Fit the model
    model.fit(X_train, y_train)
    # Predict
    y_pred = model.predict(X_test)
    # Calculate metrics
    mse = mean_squared_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)
    return mse, r2

In [13]:
# Apply the function to each model and store results
models[['mse', 'r2']] = models['model'].apply(
    lambda m: pd.Series(evaluate_model(m, X_train, X_test, y_train, y_test))
)

# Display results
print(models)

          model_name                                              model  \
0  Linear Regression                                 LinearRegression()   
1      Random Forest  (DecisionTreeRegressor(max_features=1.0, rando...   
2            XGBoost  XGBRegressor(base_score=None, booster=None, ca...   

        mse        r2  
0  0.065686  0.026825  
1  0.054538  0.191978  
2  0.084721 -0.255204  


In [14]:
from sklearn.model_selection import cross_val_score

def evaluate_model_cv(model, X, y):
    # Perform 5-fold cross-validation and return mean MSE
    scores = cross_val_score(model, X, y, cv=5, scoring='neg_mean_squared_error')
    return -np.mean(scores)

# Apply cross-validation
models['cv_mse'] = models['model'].apply(lambda m: evaluate_model_cv(m, X, y))

# Display updated results
print(models)

          model_name                                              model  \
0  Linear Regression                                 LinearRegression()   
1      Random Forest  (DecisionTreeRegressor(max_features=1.0, rando...   
2            XGBoost  XGBRegressor(base_score=None, booster=None, ca...   

        mse        r2    cv_mse  
0  0.065686  0.026825  0.080500  
1  0.054538  0.191978  0.093902  
2  0.084721 -0.255204  0.125069  
